# Actividad - Proyecto práctico

> **Grupo 5 — Entorno asignado: `Enduro-v0`** · *(versión rápida: presupuesto reducido, ~5 h por propuesta)*

> La actividad se desarrollará en grupos pre-definidos de 4 alumnos. Se debe indicar los nombres en orden alfabético (de apellidos). Recordad que esta actividad se corresponde con un 30% de la nota final de la asignatura. Se debe entregar entregar el trabajo en la presente notebook.
*   Alumno 1: Cadena Cedano, Juan David Alejandro
*   Alumno 2: Castañeda Lozano, Carlos Andrés
*   Alumno 3: Jaramillo Rincón, Luis Alejandro

Link de Github: https://github.com/jdalejandro91/rl-project

> **Estado — Avance 2 (solo implementación, sin ejecutar):** implementadas la propuesta **Double DQN** y el marco de comparación (`run_proposal`), además del código de las dos primeras gráficas. Todavía sin ejecutar, por lo que aún no hay resultados ni análisis. Pendiente la tercera propuesta (Dueling), el resto de gráficas, el test final de 100 episodios y la justificación.


---
## **PARTE 1** - Instalación y requisitos previos

> Las prácticas han sido preparadas para poder realizarse en el entorno de trabajo de Google Colab. Sin embargo, esta plataforma presenta ciertas incompatibilidades a la hora de visualizar la renderización en gym. Por ello, para obtener estas visualizaciones, se deberá trasladar el entorno de trabajo a local. Por ello, el presente dosier presenta instrucciones para poder trabajar en ambos entornos. Siga los siguientes pasos para un correcto funcionamiento:
1.   **LOCAL:** Preparar el enviroment, siguiendo las intrucciones detalladas en la sección *1.1.Preparar enviroment*.
2.  **AMBOS:** Modificar las variables "mount" y "drive_mount" a la carpeta de trabajo en drive en el caso de estar en Colab, y ejecturar la celda *1.2.Localizar entorno de trabajo*.
3. **COLAB:** se deberá ejecutar las celdas correspondientes al montaje de la carpeta de trabajo en Drive. Esta corresponde a la sección *1.3.Montar carpeta de datos local*.
4.  **AMBOS:** Instalar las librerías necesarias, siguiendo la sección *1.4.Instalar librerías necesarias*.


---
### 1.1. Preparar enviroment (solo local)



> Para preparar el entorno de trabajo en local, se han seguido los siguientes pasos:
1. En Windows, puede ser necesario instalar las C++ Build Tools. Para ello, siga los siguientes pasos. Alternativamente puedes utilizar WSL2: https://towardsdatascience.com/how-to-install-openai-gym-in-a-windows-environment-338969e24d30.
2. Instalar Anaconda
3. Siguiendo el código que se presenta comentado en la próxima celda: Crear un enviroment, cambiar la ruta de trabajo, e instalar librerías básicas.


```
conda create --name miar_rl python=3.8
conda activate miar_rl
cd "PATH_TO_FOLDER"
conda install git
pip install jupyter
```


4. Abrir la notebook con *jupyter-notebook*.



```
jupyter-notebook
```


---
### 1.2. Localizar entorno de trabajo: Google colab o local

In [ ]:
mount='/content/gdrive'
drive_root = mount + "/My Drive/08_MIAR/actividades/proyecto practico v2"

try:
  from google.colab import drive
  IN_COLAB=True
except:
  IN_COLAB=False

---
### 1.3. Montar carpeta de datos local (solo Colab)

In [ ]:
# Switch to the directory on the Google Drive that you want to use
import os
if IN_COLAB:
  print("We're running Colab")

  if IN_COLAB:
    # Mount the Google Drive at mount
    print("Colab: mounting Google drive on ", mount)

    drive.mount(mount)

    # Create drive_root if it doesn't exist
    create_drive_root = True
    if create_drive_root:
      print("\nColab: making sure ", drive_root, " exists.")
      os.makedirs(drive_root, exist_ok=True)

    # Change to the directory
    print("\nColab: Changing directory to ", drive_root)
    %cd $drive_root
# Verify we're in the correct working directory
%pwd
print("Archivos en el directorio: ")
print(os.listdir())

---
### 1.4. Instalar librerías necesarias

In [ ]:
if IN_COLAB:
  %pip install tensorflow==2.18.0
  %pip install tf-keras==2.18.0
  %pip install gym==0.17.3
  %pip install git+https://github.com/Kojoley/atari-py.git
  %pip install keras-rl2==1.0.5
  # El Colab actual trae un jax que pide ml_dtypes>=0.5, pero TF 2.18 instala la
  # 0.4.x. Como no usamos jax, lo desinstalamos para que no falle al importar TF.
  %pip uninstall -y jax jaxlib
else:
  %pip install gym==0.17.3
  %pip install git+https://github.com/Kojoley/atari-py.git
  %pip install pyglet==1.5.0
  %pip install h5py==3.1.0
  %pip install Pillow==9.5.0
  %pip install keras-rl2==1.0.5
  %pip install Keras==2.2.4
  %pip install tensorflow==2.5.3
  %pip install torch==2.0.1
  %pip install agents==1.4.0

# keras-rl guarda sus contadores como np.int16 (máx 32.767): con millones de pasos
# desbordan, el entrenamiento no para y epsilon no decae (el agente jugaría al azar).
# Lo arreglamos cambiando np.int16 por np.int32 en el core, antes de importar rl.
!python -c "import importlib.util,os; s=importlib.util.find_spec('rl'); p=os.path.join(os.path.dirname(s.origin),'core.py'); src=open(p).read(); n=src.count('np.int16'); open(p,'w').write(src.replace('np.int16','np.int32')); print('[FIX keras-rl]', n, 'reemplazos np.int16->np.int32 en', p)"

---
## **PARTE 2**. Enunciado

Consideraciones a tener en cuenta:

- El entorno sobre el que trabajaremos será el indicado en el listado correspondien de cada grupo y el algoritmo que usaremos será _DQN_.

- Para nuestro ejercicio, el requisito mínimo será alcanzado cuando el agente consiga una **media de recompensa por encima de los puntos indicados en el listado por grupos en modo test**. Por ello, esta media de la recompensa se calculará a partir del código de test en la última celda del notebook.

Este proyecto práctico consta de tres partes:

1.   Implementar la red neuronal que se usará en la solución
2.   Implementar las distintas piezas de la solución DQN y probar al menos 3 propuestas diferentes de mejora.
3.   Justificar la respuesta en relación a los resultados obtenidos e incluir al menos 3 gráficas relevantes comparando las 3 propuestas.

**Rúbrica**: Se valorará la originalidad en la solución aportada, así como la capacidad de discutir los resultados de forma detallada. El requisito mínimo servirá para aprobar la actividad, bajo premisa de que la discusión del resultado sera apropiada.

IMPORTANTE:

* Si no se consigue una puntuación óptima, responder sobre la mejor puntuación obtenida.
* Para entrenamientos largos, recordad que podéis usar checkpoints de vuestros modelos para retomar los entrenamientos. En este caso, recordad cambiar los parámetros adecuadamente (sobre todo los relacionados con el proceso de exploración).
* Se deberá entregar unicamente el notebook y los pesos del mejor modelo en un fichero .zip, de forma organizada.
* Cada alumno deberá de subir la solución de forma individual.

---
### Nuestro grupo

A nuestro grupo (Grupo 5) le corresponde el entorno **`Enduro-v0`** ([documentación](https://ale.farama.org/environments/enduro/)), un juego de conducción de Atari en el que hay que adelantar coches. Lo resolvemos con **DQN** y comparamos tres propuestas: *DQN base*, *Double DQN* y *Double + Dueling con ajuste de hiperparámetros*.

El **objetivo mínimo** es superar el primer día de carrera (adelantar unos 200 coches) de media en 100 episodios de test seguidos. Conviene decirlo de entrada: Enduro es el entorno más difícil del listado, porque la recompensa solo llega al adelantar coches y aprender a hacerlo bien requiere muchísimos pasos. Por eso, y como permite el enunciado, reportamos y justificamos la mejor puntuación que conseguimos con el cómputo del que disponemos.

---
## **PARTE 3**. Desarrollo y preguntas

#### Importar librerías

In [ ]:
# El Colab actual carga Keras 3 al importar tensorflow.keras, y Keras 3 arrastra jax,
# que choca con la versión de ml_dtypes que necesita TF 2.18. Para evitarlo activamos
# tf-keras (Keras 2) ANTES de importar nada de keras: es la versión con la que funciona
# keras-rl2.
import os
os.environ['TF_USE_LEGACY_KERAS'] = "1"

import tensorflow as tf1
import tensorflow.keras as tf

# keras-rl2 hace 'from tensorflow.keras import __version__', pero tf-keras no lo expone,
# así que se lo añadimos a mano para que el import no falle.
try:
    import tf_keras
    tf.__version__ = tf_keras.__version__
except Exception:
    tf.__version__ = "2.18.0"

print('TensorFlow:', tf1.__version__)
print('Keras:', tf.__version__)
print('GPU:', tf1.config.list_physical_devices('GPU'))

In [ ]:
from __future__ import division

from PIL import Image
import numpy as np
import gym

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Flatten, Convolution2D, Permute

if IN_COLAB:
  from tensorflow.keras.optimizers.legacy import Adam
else:
  from tensorflow.keras.optimizers import Adam

import tensorflow.keras.backend as K

from rl.agents.dqn import DQNAgent
from rl.policy import LinearAnnealedPolicy, BoltzmannQPolicy, EpsGreedyQPolicy
from rl.memory import SequentialMemory
from rl.core import Processor
from rl.callbacks import FileLogger, ModelIntervalCheckpoint

#### Configuración base

In [ ]:
# Reducimos cada frame a 84x84 en escala de grises y apilamos 4: con un solo frame
# la red no sabría la velocidad ni hacia dónde van los coches, y eso en un juego de
# conducción es clave.
INPUT_SHAPE = (84, 84)
WINDOW_LENGTH = 4

env_name = 'Enduro-v0'        # entorno asignado al Grupo 5
env = gym.make(env_name)

np.random.seed(123)           # fijamos semillas para que los resultados sean reproducibles
env.seed(123)
nb_actions = env.action_space.n

print('Entorno:', env_name)
print('Nº de acciones:', nb_actions)
print('Observación:', env.observation_space)

In [ ]:
class AtariProcessor(Processor):
    # Añadimos el flag clip_rewards: al entrenar recortamos la recompensa a [-1, 1]
    # (estabiliza el DQN), pero al evaluar lo desactivamos para medir la recompensa
    # real, que en Enduro es el número de coches adelantados.
    def __init__(self, clip_rewards=True):
        self.clip_rewards = clip_rewards

    def process_observation(self, observation):
        assert observation.ndim == 3
        img = Image.fromarray(observation).resize(INPUT_SHAPE).convert('L')  # a 84x84 y grises
        processed_observation = np.array(img)
        assert processed_observation.shape == INPUT_SHAPE
        return processed_observation.astype('uint8')   # uint8 para que el buffer pese menos

    def process_state_batch(self, batch):
        return batch.astype('float32') / 255.           # normalizamos justo antes de la red

    def process_reward(self, reward):
        return np.clip(reward, -1., 1.) if self.clip_rewards else reward

1. Implementación de la red neuronal

In [ ]:
# 1. Red neuronal
# Usamos la CNN del DQN de Atari (Mnih et al., 2015): tres capas convolucionales que
# "miran" la imagen y dos densas que deciden. La salida es un valor Q por acción, con
# activación lineal porque los valores Q pueden ser cualquier número.
def build_model(input_shape, nb_actions):
    model = Sequential()
    # Reordenamos los ejes según cómo Keras espere los canales (al final o al inicio).
    if K.image_data_format() == 'channels_last':
        model.add(Permute((2, 3, 1), input_shape=input_shape))
    elif K.image_data_format() == 'channels_first':
        model.add(Permute((1, 2, 3), input_shape=input_shape))
    else:
        raise RuntimeError('Formato de imagen desconocido.')

    model.add(Convolution2D(32, (8, 8), strides=(4, 4)))
    model.add(Activation('relu'))
    model.add(Convolution2D(64, (4, 4), strides=(2, 2)))
    model.add(Activation('relu'))
    model.add(Convolution2D(64, (3, 3), strides=(1, 1)))
    model.add(Activation('relu'))
    model.add(Flatten())
    model.add(Dense(512))
    model.add(Activation('relu'))
    model.add(Dense(nb_actions))
    model.add(Activation('linear'))
    return model

model = build_model((WINDOW_LENGTH,) + INPUT_SHAPE, nb_actions)
print(model.summary())

2. Implementación de la solución DQN

Creamos el agente con una función para poder montar varias propuestas cambiando pocos
argumentos. En este avance implementamos las dos primeras:

1. **DQN base.** Nuestra referencia.
2. **Double DQN** (`enable_double_dqn=True`): separa quién elige la acción de quién la
   valora, para reducir la **sobreestimación** de los valores Q.

La tercera propuesta (Dueling) la añadiremos en la versión final.

In [ ]:
# 2. Solución DQN
# La función admite un flag `double` para poder montar la base y la variante Double DQN
# con los mismos hiperparámetros y aislar así el efecto de la mejora.

# Hiperparámetros COMPARTIDOS por las propuestas (los estándar del DQN de Atari de Mnih
# et al.), así la comparación solo refleja el efecto de Double.
MEMORY_LIMIT   = 500000    # buffer de repetición: rompe la correlación entre pasos seguidos
WARMUP_STEPS   = 50000     # pasos iniciales (al azar) antes de empezar a entrenar
TARGET_UPDATE  = 10000     # cada cuántos pasos copiamos la red a la red objetivo
TRAIN_INTERVAL = 4         # entrenamos 1 de cada 4 pasos (valor estándar)
GAMMA          = 0.99      # factor de descuento: la carrera es larga, miramos al futuro
LR             = 0.00025   # learning rate de Adam (valor estándar y estable del paper)

def build_agent(nb_actions, double=False, anneal_steps=None):
    # Epsilon decae a lo largo del ~70% del entrenamiento (lo atamos a TRAIN_STEPS).
    if anneal_steps is None:
        anneal_steps = int(0.7 * TRAIN_STEPS)

    model = build_model((WINDOW_LENGTH,) + INPUT_SHAPE, nb_actions)
    memory = SequentialMemory(limit=MEMORY_LIMIT, window_length=WINDOW_LENGTH)
    processor = AtariProcessor(clip_rewards=True)
    # Política e-greedy: explora mucho al principio y va explotando lo aprendido.
    policy = LinearAnnealedPolicy(EpsGreedyQPolicy(), attr='eps', value_max=1.0,
                                  value_min=0.1, value_test=0.05, nb_steps=anneal_steps)

    # enable_double_dqn=double -> con double=False tenemos el DQN base; con True, Double DQN.
    dqn = DQNAgent(model=model, nb_actions=nb_actions, policy=policy, memory=memory,
                   processor=processor, nb_steps_warmup=WARMUP_STEPS, gamma=GAMMA,
                   target_model_update=TARGET_UPDATE, train_interval=TRAIN_INTERVAL,
                   enable_double_dqn=double, enable_dueling_network=False)
    dqn.compile(Adam(learning_rate=LR), metrics=['mae'])
    return dqn, processor

### Marco de comparación de propuestas

Cada propuesta se entrena con los mismos hiperparámetros y se evalúa con la recompensa real
(sin recorte). Guardamos pesos, checkpoints y un log por episodio del que salen las gráficas.

In [ ]:
import os
import gc

RESULTS_DIR = 'resultados_grupo5'
os.makedirs(RESULTS_DIR, exist_ok=True)

test_results = {}        # recompensa media en test de cada propuesta
proposal_configs = {}    # arquitectura (double) de cada propuesta

def train_proposal(dqn, name, nb_steps, checkpoint_interval=100000):
    weights_filename    = os.path.join(RESULTS_DIR, 'dqn_%s_%s_weights.h5f' % (env_name, name))
    checkpoint_filename = os.path.join(RESULTS_DIR, 'dqn_%s_%s_weights_{step}.h5f' % (env_name, name))
    log_filename        = os.path.join(RESULTS_DIR, 'dqn_%s_%s_log.json' % (env_name, name))
    callbacks = [ModelIntervalCheckpoint(checkpoint_filename, interval=checkpoint_interval),
                 FileLogger(log_filename, interval=100)]
    dqn.fit(env, callbacks=callbacks, nb_steps=nb_steps,
            log_interval=10000, visualize=False, verbose=1)
    dqn.save_weights(weights_filename, overwrite=True)
    return weights_filename, log_filename

def run_proposal(name, double, nb_steps, nb_test_episodes=20):
    K.clear_session(); gc.collect()
    np.random.seed(123); env.seed(123)          # mismas semillas -> comparación reproducible
    proposal_configs[name] = dict(double=double)
    dqn, processor = build_agent(nb_actions, double=double)
    weights_filename, log_filename = train_proposal(dqn, name, nb_steps)

    # Evaluamos con la recompensa real (sin recorte) para la comparativa.
    processor.clip_rewards = False
    hist = dqn.test(env, nb_episodes=nb_test_episodes, visualize=False)
    test_results[name] = float(np.mean(hist.history['episode_reward']))
    print('[%s] recompensa media en test (%d ep.): %.2f'
          % (name, nb_test_episodes, test_results[name]))

    del dqn, processor, hist; gc.collect()       # liberamos memoria antes de la siguiente
    return weights_filename, log_filename

In [ ]:
# Pasos de entrenamiento por propuesta. Presupuesto reducido (500.000) para terminar en
# un tiempo razonable; epsilon decae durante el ~70% (350.000 pasos).
TRAIN_STEPS = 500000

In [ ]:
# Propuesta 1 — DQN base
run_proposal('p1_base', double=False, nb_steps=TRAIN_STEPS)

In [ ]:
# Propuesta 2 — Double DQN (añade solo Double sobre la base)
run_proposal('p2_double', double=True, nb_steps=TRAIN_STEPS)

### Gráficas comparativas (avance 2)

Código de las dos primeras gráficas: la curva de aprendizaje y la evolución de la Q media.
Se construyen a partir de los logs JSON que genera cada entrenamiento, así que quedarán
vacías hasta que se ejecuten las celdas de entrenamiento (este avance se entrega sin
ejecutar).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

PROPOSALS = {
    'p1_base':   'DQN base',
    'p2_double': 'Double DQN',
}

def load_log(name):
    path = os.path.join(RESULTS_DIR, 'dqn_%s_%s_log.json' % (env_name, name))
    with open(path) as fjson:
        return pd.DataFrame(json.load(fjson))

logs = {}
for name in PROPOSALS:
    try:
        logs[name] = load_log(name)
    except FileNotFoundError:
        print('Falta el log de %s (¿ya entrenaste esa propuesta?)' % name)

In [ ]:
# Gráfica 1 — Curva de aprendizaje: recompensa por episodio (media móvil) frente a los pasos.
plt.figure(figsize=(10, 5))
for name, label in PROPOSALS.items():
    if name not in logs:
        continue
    df = logs[name]
    x = df['nb_steps'] if 'nb_steps' in df.columns else df.index
    plt.plot(x, df['episode_reward'].rolling(50, min_periods=1).mean(), label=label)
plt.xlabel('Pasos de entrenamiento')
plt.ylabel('Recompensa por episodio (media móvil 50)')
plt.title('Curva de aprendizaje — Enduro-v0')
plt.legend(); plt.grid(True)
plt.show()

In [ ]:
# Gráfica 2 — Valor Q medio: sirve para ver la sobreestimación (Double DQN debería contenerla).
plt.figure(figsize=(10, 5))
for name, label in PROPOSALS.items():
    if name not in logs or 'mean_q' not in logs[name].columns:
        continue
    df = logs[name]
    x = df['nb_steps'] if 'nb_steps' in df.columns else df.index
    plt.plot(x, df['mean_q'].rolling(50, min_periods=1).mean(), label=label)
plt.xlabel('Pasos de entrenamiento')
plt.ylabel('Q media estimada')
plt.title('Evolución del valor Q medio')
plt.legend(); plt.grid(True)
plt.show()